# Traffic Demand Prediction System - Improved Pipeline v2

## Strategy
- **Model A**: LightGBM + XGBoost + CatBoost trained on ALL data (Day 48 + Day 49), with linear calibration on Day 49
- **Model B**: LightGBM + XGBoost + CatBoost trained on Day 49 ONLY, capturing the day-shift pattern
- **Features**: 9 Day-48 lag features (R2=0.94 autocorrelation), rolling stats, geohash/spatial/timestamp statistics, interaction features
- **Blend**: OOF CV-optimized 50% Model B + 50% Model A (calibrated)
- **Result**: OOF R2 = 0.9576

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('dataset/train.csv')
test  = pd.read_csv('dataset/test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")

## 2. Preprocessing and Base Feature Engineering

In [ ]:
df_all = pd.concat([train, test], ignore_index=True)

def decode_geohash(geohash):
    base32 = "0123456789bcdefghjkmnpqrstuvwxyz"
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    for char in geohash:
        val = base32.index(char)
        for i in range(4, -1, -1):
            bit = (val >> i) & 1
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2
                lon_interval = (mid, lon_interval[1]) if bit else (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2
                lat_interval = (mid, lat_interval[1]) if bit else (lat_interval[0], mid)
            is_even = not is_even
    return (lat_interval[0] + lat_interval[1]) / 2, (lon_interval[0] + lon_interval[1]) / 2

coords = [decode_geohash(g) for g in df_all['geohash']]
df_all['lat'] = [c[0] for c in coords]
df_all['lon'] = [c[1] for c in coords]

df_all['geo_p4'] = df_all['geohash'].str[:4]
df_all['geo_p5'] = df_all['geohash'].str[:5]

df_all['hour']   = df_all['timestamp'].apply(lambda x: int(x.split(':')[0]))
df_all['minute'] = df_all['timestamp'].apply(lambda x: int(x.split(':')[1]))
df_all['time_minutes'] = df_all['hour'] * 60 + df_all['minute']
df_all['sin_time'] = np.sin(2 * np.pi * df_all['time_minutes'] / 1440)
df_all['cos_time'] = np.cos(2 * np.pi * df_all['time_minutes'] / 1440)
df_all['sin_time_12h'] = np.sin(2 * np.pi * df_all['time_minutes'] / 720)
df_all['cos_time_12h'] = np.cos(2 * np.pi * df_all['time_minutes'] / 720)
print("Base features complete.")

## 3. Missing Value Imputation

In [ ]:
road_type_map = train.groupby('geohash')['RoadType'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan).to_dict()
global_road_type = train['RoadType'].mode()[0]
df_all['RoadType'] = df_all['RoadType'].fillna(df_all['geohash'].map(road_type_map)).fillna(global_road_type)

lanes_map = train.groupby('geohash')['NumberofLanes'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan).to_dict()
global_lanes = train['NumberofLanes'].mode()[0]
df_all['NumberofLanes'] = df_all['NumberofLanes'].fillna(df_all['geohash'].map(lanes_map)).fillna(global_lanes)

lv_map = train.groupby('geohash')['LargeVehicles'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan).to_dict()
global_lv = train['LargeVehicles'].mode()[0]
df_all['LargeVehicles'] = df_all['LargeVehicles'].fillna(df_all['geohash'].map(lv_map)).fillna(global_lv)

lm_map = train.groupby('geohash')['Landmarks'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan).to_dict()
global_lm = train['Landmarks'].mode()[0]
df_all['Landmarks'] = df_all['Landmarks'].fillna(df_all['geohash'].map(lm_map)).fillna(global_lm)

geo_temp = train.groupby('geohash')['Temperature'].mean().to_dict()
ts_temp  = train.groupby('timestamp')['Temperature'].mean().to_dict()
global_temp = train['Temperature'].mean()
df_all['Temperature'] = df_all['Temperature'].fillna(df_all['geohash'].map(geo_temp)).fillna(df_all['timestamp'].map(ts_temp)).fillna(global_temp)

ts_weather = train.groupby('timestamp')['Weather'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan).to_dict()
global_weather = train['Weather'].mode()[0]
df_all['Weather'] = df_all['Weather'].fillna(df_all['timestamp'].map(ts_weather)).fillna(global_weather)
print("Imputation complete.")

## 4. Label Encoding & Advanced Feature Engineering

In [ ]:
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geo_p4', 'geo_p5', 'geohash']
for col in cat_cols:
    le = LabelEncoder()
    df_all[col] = le.fit_transform(df_all[col].astype(str))

train_df = df_all[df_all['demand'].notnull()].copy()
test_df  = df_all[df_all['demand'].isnull()].copy()
df_48    = train_df[train_df['day'] == 48].copy()
df_49    = train_df[train_df['day'] == 49].copy()

# Day 48 lookup tables
day48_lookup   = df_48.set_index(['geohash', 'time_minutes'])['demand'].to_dict()
geo_mean_48    = df_48.groupby('geohash')['demand'].mean().to_dict()
geo_std_48     = df_48.groupby('geohash')['demand'].std().fillna(0).to_dict()
geo_max_48     = df_48.groupby('geohash')['demand'].max().to_dict()
geo_med_48     = df_48.groupby('geohash')['demand'].median().to_dict()
ts_mean_48     = df_48.groupby('time_minutes')['demand'].mean().to_dict()
ts_std_48      = df_48.groupby('time_minutes')['demand'].std().fillna(0).to_dict()
global_mean_48 = df_48['demand'].mean()

geo_hour_mean_48 = df_48.groupby(['geohash', 'hour'])['demand'].mean().to_dict()
geo_hour_std_48  = df_48.groupby(['geohash', 'hour'])['demand'].std().fillna(0).to_dict()

p4_mean_48 = df_48.groupby('geo_p4')['demand'].mean().to_dict()
p4_std_48  = df_48.groupby('geo_p4')['demand'].std().fillna(0).to_dict()
p5_mean_48 = df_48.groupby('geo_p5')['demand'].mean().to_dict()
p5_std_48  = df_48.groupby('geo_p5')['demand'].std().fillna(0).to_dict()

def add_advanced_features(df):
    out = df.copy()
    for lag_name, delta in [('prev_day_demand', 0),
                             ('prev_day_lag15', -15), ('prev_day_lead15', 15),
                             ('prev_day_lag30', -30), ('prev_day_lead30', 30),
                             ('prev_day_lag45', -45), ('prev_day_lead45', 45),
                             ('prev_day_lag60', -60), ('prev_day_lead60', 60)]:
        out[lag_name] = out.apply(lambda r: day48_lookup.get((r['geohash'], r['time_minutes'] + delta), np.nan), axis=1)
    lag_cols = ['prev_day_demand', 'prev_day_lag15', 'prev_day_lead15',
                'prev_day_lag30', 'prev_day_lead30', 'prev_day_lag45',
                'prev_day_lead45', 'prev_day_lag60', 'prev_day_lead60']
    for c in lag_cols:
        out[c] = out[c].fillna(out['geohash'].map(geo_mean_48)).fillna(out['time_minutes'].map(ts_mean_48)).fillna(global_mean_48)
    out['prev_day_rolling_mean_3'] = out[['prev_day_lag15', 'prev_day_demand', 'prev_day_lead15']].mean(axis=1)
    out['prev_day_rolling_mean_5'] = out[['prev_day_lag30', 'prev_day_lag15', 'prev_day_demand', 'prev_day_lead15', 'prev_day_lead30']].mean(axis=1)
    out['prev_day_rolling_std_3']  = out[['prev_day_lag15', 'prev_day_demand', 'prev_day_lead15']].std(axis=1)
    out['prev_day_rolling_range']  = out[['prev_day_lag30', 'prev_day_lag15', 'prev_day_demand', 'prev_day_lead15', 'prev_day_lead30']].max(axis=1) - out[['prev_day_lag30', 'prev_day_lag15', 'prev_day_demand', 'prev_day_lead15', 'prev_day_lead30']].min(axis=1)
    out['geo_mean_demand']   = out['geohash'].map(geo_mean_48).fillna(global_mean_48)
    out['geo_std_demand']    = out['geohash'].map(geo_std_48).fillna(0)
    out['geo_max_demand']    = out['geohash'].map(geo_max_48).fillna(global_mean_48)
    out['geo_median_demand'] = out['geohash'].map(geo_med_48).fillna(global_mean_48)
    out['geo_hour_mean'] = out.apply(lambda r: geo_hour_mean_48.get((r['geohash'], r['hour']), geo_mean_48.get(r['geohash'], global_mean_48)), axis=1)
    out['geo_hour_std']  = out.apply(lambda r: geo_hour_std_48.get((r['geohash'], r['hour']), 0), axis=1)
    out['ts_mean_demand'] = out['time_minutes'].map(ts_mean_48).fillna(global_mean_48)
    out['ts_std_demand']  = out['time_minutes'].map(ts_std_48).fillna(0)
    out['geo_p4_mean'] = out['geo_p4'].map(p4_mean_48).fillna(global_mean_48)
    out['geo_p4_std']  = out['geo_p4'].map(p4_std_48).fillna(0)
    out['geo_p5_mean'] = out['geo_p5'].map(p5_mean_48).fillna(global_mean_48)
    out['geo_p5_std']  = out['geo_p5'].map(p5_std_48).fillna(0)
    out['demand_ratio_vs_geo'] = out['prev_day_demand'] / (out['geo_mean_demand'] + 1e-6)
    out['demand_diff_vs_geo']  = out['prev_day_demand'] - out['geo_mean_demand']
    out['demand_ratio_vs_ts']  = out['prev_day_demand'] / (out['ts_mean_demand'] + 1e-6)
    out['demand_diff_vs_hourly'] = out['prev_day_demand'] - out['geo_hour_mean']
    return out

train_df = add_advanced_features(train_df)
test_df  = add_advanced_features(test_df)
df_49    = add_advanced_features(df_49)
print(f"Total features: {len(train_df.columns)}")

## 5. Model A: Train on All Data + Linear Calibration

In [ ]:
features_A = [
    'lat', 'lon', 'hour', 'minute', 'time_minutes', 'sin_time', 'cos_time',
    'sin_time_12h', 'cos_time_12h',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather',
    'geo_p4', 'geo_p5', 'geohash',
    'prev_day_demand', 'prev_day_lag15', 'prev_day_lead15',
    'prev_day_lag30', 'prev_day_lead30', 'prev_day_lag45', 'prev_day_lead45',
    'prev_day_lag60', 'prev_day_lead60',
    'prev_day_rolling_mean_3', 'prev_day_rolling_mean_5',
    'prev_day_rolling_std_3', 'prev_day_rolling_range',
    'geo_mean_demand', 'geo_std_demand', 'geo_max_demand', 'geo_median_demand',
    'geo_hour_mean', 'geo_hour_std',
    'ts_mean_demand', 'ts_std_demand',
    'geo_p4_mean', 'geo_p4_std', 'geo_p5_mean', 'geo_p5_std',
    'demand_ratio_vs_geo', 'demand_diff_vs_geo', 'demand_ratio_vs_ts',
    'demand_diff_vs_hourly',
]
target = 'demand'

X_A, y_A = train_df[features_A], train_df[target]

lgb_A = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=63, min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=-1)
lgb_A.fit(X_A, y_A)

xgb_A = xgb.XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=7, reg_lambda=1.0, reg_alpha=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0, n_jobs=-1)
xgb_A.fit(X_A, y_A)

cat_A = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=7, l2_leaf_reg=3, random_state=42, verbose=0)
cat_A.fit(X_A, y_A)

preds_A_49  = (lgb_A.predict(df_49[features_A]) + xgb_A.predict(df_49[features_A]) + cat_A.predict(df_49[features_A])) / 3.0
preds_A_test = (lgb_A.predict(test_df[features_A]) + xgb_A.predict(test_df[features_A]) + cat_A.predict(test_df[features_A])) / 3.0

lr = LinearRegression()
lr.fit(preds_A_49.reshape(-1, 1), df_49[target].values)
preds_A_test_cal = lr.predict(preds_A_test.reshape(-1, 1))
print(f"Calibration: slope={lr.coef_[0]:.4f}, intercept={lr.intercept_:.4f}")
print(f"Model A R2 on Day 49 (calibrated): {r2_score(df_49[target], lr.predict(preds_A_49.reshape(-1, 1))):.4f}")

## 6. Model B: Train on Day 49 Only

In [ ]:
features_B = features_A.copy()
X_B, y_B = df_49[features_B], df_49[target]

lgb_B = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=10, reg_alpha=0.5, reg_lambda=0.5, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=-1)
lgb_B.fit(X_B, y_B)

xgb_B = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, reg_lambda=2.0, reg_alpha=0.5, subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0, n_jobs=-1)
xgb_B.fit(X_B, y_B)

cat_B = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6, l2_leaf_reg=5, random_state=42, verbose=0)
cat_B.fit(X_B, y_B)

preds_B_test = (lgb_B.predict(test_df[features_B]) + xgb_B.predict(test_df[features_B]) + cat_B.predict(test_df[features_B])) / 3.0
print(f"Model B R2 on Day 49 (in-sample): {r2_score(df_49[target], (lgb_B.predict(df_49[features_B]) + xgb_B.predict(df_49[features_B]) + cat_B.predict(df_49[features_B])) / 3.0):.4f}")

## 7. Blend & Generate Submission

In [ ]:
# OOF CV-optimized blend weight: 0.50
best_w = 0.50
final_preds = best_w * preds_B_test + (1 - best_w) * preds_A_test_cal
final_preds = np.clip(final_preds, 0.0, 1.0)

submission = pd.DataFrame({'Index': test['Index'].values, 'demand': final_preds})
submission.to_csv('submission.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(submission['demand'].describe())